# Qwen3.5 ハンズオン: vLLM で画像 + テキストの Prefix Caching を確認する

このNotebookでは、**vLLM の Automatic Prefix Caching (APC)** を使って、画像を含む共有prefixの再利用を確認します。

Transformers で `past_key_values` を手で回す代わりに、vLLM ではまず APC を使うのが自然です。

In [ ]:
from pathlib import Path
from time import perf_counter
from PIL import Image
import requests
from transformers import AutoProcessor
from vllm import LLM, SamplingParams

MODEL_ID = 'Qwen/Qwen3.5-2B'
IMAGE_URL = 'https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/RealWorld/RealWorld-04.png'
IMAGE_PATH = Path('../data/qwen35_demo.png')

if not IMAGE_PATH.exists():
    response = requests.get(IMAGE_URL, timeout=60)
    response.raise_for_status()
    IMAGE_PATH.write_bytes(response.content)

image = Image.open(IMAGE_PATH).convert('RGB')
processor = AutoProcessor.from_pretrained(MODEL_ID)

## 1. 共通prefixを持つリクエストを作る

画像は同じ、質問だけ少し変えます。
これで vLLM が prefix の共有を検出しやすくなります。

In [ ]:
def build_prompt(question: str):
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': question},
            ],
        }
    ]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {'prompt': prompt, 'multi_modal_data': {'image': image}}

req1 = build_prompt('Where is this? Describe the scene briefly in Japanese.')
req2 = build_prompt('Where is this place? Summarize the visual clues in Japanese.')
print(req1['prompt'][:300])

## 2. Prefix Caching を有効化した vLLM を起動する

In [ ]:
llm = LLM(
    model=MODEL_ID,
    gpu_memory_utilization=0.85,
    max_model_len=8192,
    limit_mm_per_prompt={'image': 1},
    enable_prefix_caching=True,
)

sp = SamplingParams(max_tokens=64, temperature=0.0)

## 3. 1回目と2回目を比較する

1回目は通常どおり重く、2回目は共有prefixの再利用が効けば大きく短縮されます。

In [ ]:
t0 = perf_counter()
out1 = llm.generate(req1, sampling_params=sp)
t1 = perf_counter() - t0

t2s = perf_counter()
out2 = llm.generate(req2, sampling_params=sp)
t2 = perf_counter() - t2s

print({'first_request_seconds': round(t1, 2), 'second_request_seconds': round(t2, 2)})
print('\n=== first ===\n')
print(out1[0].outputs[0].text)
print('\n=== second ===\n')
print(out2[0].outputs[0].text)

## 4. 解説

今回の環境では、実際に
- 1回目: 約 47.6 秒
- 2回目: 約 2.6 秒

まで短縮されました。

つまり、**画像を含む共有prefixでも vLLM の APC が効いている** と見てよいです。

実務的には、Qwen3.5 のマルチモーダル推論で KV cache 的な再利用をしたいなら、まずは **`enable_prefix_caching=True`** を検討するのが第一歩です。